In [48]:
import pandas as pd
import numpy as np
import ollama
import warnings
warnings.filterwarnings("ignore")

In [49]:
ollama.pull('qwen2.5:1.5b')

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [50]:
# Load exercise dataset
df_exercises = pd.read_csv("megaGymDataset.csv")
df_exercises.head()

,Unnamed: 0,Title,Desc,Type,BodyPart,Equipment,Level,Rating,RatingDesc
0,0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0,NaN
1,1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
2,2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
3,3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
4,4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,NaN,NaN


In [51]:
# Check shape and missing values
print("Shape:", df_exercises.shape)
print("\nMissing values:")
print(df_exercises.isnull().sum())

Shape: (2918, 9)

Missing values:
Unnamed: 0       0
Title            0
Desc          1550
Type             0
BodyPart         0
Equipment       32
Level            0
Rating        1887
RatingDesc    2056
dtype: int64


In [52]:
df_exercises = df_exercises.drop(columns=["Unnamed: 0", "Rating", "RatingDesc"])

In [53]:
df_exercises = df_exercises.dropna(subset = ['Desc'])

In [54]:
df_exercises = df_exercises.fillna({'Equipment': 'Unknown'})

In [55]:
print("Shape:", df_exercises.shape)
print("\nMissing values:")
df_exercises.isnull().sum()

Shape: (1368, 6)

Missing values:


Title        0
Desc         0
Type         0
BodyPart     0
Equipment    0
Level        0
dtype: int64

In [56]:
df_exercises.head()

,Title,Desc,Type,BodyPart,Equipment,Level
0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate
1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate
2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate
3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate
4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate


In [57]:
# ============================================================
# FITFUEL - Cell 4: Create Description Column
# ============================================================

df_exercises["description"] = (
    "Exercise: " + df_exercises['Title'] +
    " | Type: " + df_exercises['Type'] +
    " | Body Part: " + df_exercises['BodyPart'] +
    " | Equipment: " + df_exercises['Equipment'] +
    " | Level: " + df_exercises['Level'] +
    " | Instructions: " + df_exercises['Desc'] 
)
df_exercises.head()

,Title,Desc,Type,BodyPart,Equipment,Level,description
0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,Exercise: Partner plank band row | Type: Stren...
1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,Exercise: Banded crunch isometric hold | Type:...
2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,Exercise: FYR Banded Plank Jack | Type: Streng...
3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,Exercise: Banded crunch | Type: Strength | Bod...
4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,Exercise: Crunch | Type: Strength | Body Part:...


In [58]:
print(df_exercises["description"].iloc[2])

Exercise: FYR Banded Plank Jack | Type: Strength | Body Part: Abdominals | Equipment: Bands | Level: Intermediate | Instructions: The banded plank jack is a variation on the plank that involves moving the legs in and out for repetitions. Having a band around the thighs forces the lower body to work harder, particularly the hips and glutes. The plank jack is commonly performed as part of a bodyweight circuit, or as part of a dynamic warm-up.


In [59]:
from sentence_transformers import SentenceTransformer

exercise_descriptions = df_exercises["description"].tolist()

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings for all exercises...")
embeddings = embedding_model.encode(exercise_descriptions, show_progress_bar=True)

print(f"\nCreated {len(embeddings)} embeddings, each with {embeddings[0].shape[0]} numbers")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings for all exercises...


Batches:   0%|          | 0/43 [00:00<?, ?it/s]


Created 1368 embeddings, each with 384 numbers


In [60]:
import chromadb

client = chromadb.Client()

try:
    client.delete_collection(name="exercise_collection")
except:
    pass

exercise_collection = client.get_or_create_collection(name="exercise_collection")

In [61]:
ids        = df_exercises.index.astype(str).tolist()
documents  = df_exercises["description"].tolist()
metadatas  = df_exercises[["Type", "BodyPart", "Level"]].to_dict(orient="records")
emb_list   = [emb.tolist() for emb in embeddings]

exercise_collection.add(
    ids        = ids,
    documents  = documents,
    metadatas  = metadatas,
    embeddings = emb_list,
)

print(f"Loaded {exercise_collection.count()} exercises into ChromaDB.")

Loaded 1368 exercises into ChromaDB.


In [62]:

def retrieve_exercises(query, k=5, filter_metadata=None):
    """
    Search for the k exercises most similar to the query.
    Optionally filter by Type, BodyPart, or Level.
    """
    # Convert query to embedding
    query_embedding = embedding_model.encode([query])

    # Search ChromaDB
    results = exercise_collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k,
        where=filter_metadata if filter_metadata else None
    )

    descriptions = results["documents"][0]
    ids          = [int(i) for i in results["ids"][0]]
    rows         = df_exercises.loc[ids]

    return descriptions, rows

# Quick test
descriptions, dfMatches = retrieve_exercises("upper body strength workout for beginners", k=5)
dfMatches[["Title", "BodyPart", "Level", "Type"]]

,Title,BodyPart,Level,Type
998,Push-Up - Gethin Variation,Chest,Intermediate,Strength
1003,Push-up,Chest,Intermediate,Strength
1004,UP Push-up,Chest,Intermediate,Strength
1044,1.5-rep push-up,Chest,Intermediate,Strength
960,Pushups,Chest,Intermediate,Strength


In [63]:
def generate_answer(query, context):
    """
    Given a user query and retrieved exercise descriptions,
    ask the LLM to generate a personalized workout recommendation.
    """
    prompt = f"""
    You are FitFuel, a personalized fitness assistant. 
    Based ONLY on the following exercise descriptions, recommend a workout plan for the user.
    Suggest 3-5 exercises from the list, explain briefly why each is relevant to the query.
    Write in an encouraging and motivating tone.
    If the exercises in the list do not match the query at all, say "No matching exercises found."
    Do NOT invent exercises that are not in the list.

    EXERCISE DESCRIPTIONS:
    {context}

    USER REQUEST: {query}
    """

    response = ollama.generate(
        model="qwen2.5:1.5b",
        prompt=prompt,
        options={'temperature': 0.1}
    )

    return response['response']

In [64]:
def rag_query_exercises(query, k=5, filter_metadata=None):
    """
    Full RAG pipeline: Retrieve → Augment → Generate.
    """
    # Step 1: Retrieve
    descriptions, rows = retrieve_exercises(query, k, filter_metadata)

    # Step 2: Build context
    context = "\n\n".join(descriptions) if descriptions else "empty"

    # Step 3: Generate
    answer = generate_answer(query, context)

    return answer, rows

# Test it!
answer, dfMatches = rag_query_exercises("I want a beginner chest workout", k=5)
print("FitFuel Recommendation:\n", answer)
print("\nMatched exercises:")
dfMatches[["Title", "BodyPart", "Level", "Type"]]

FitFuel Recommendation:
 Absolutely! Let's create a fantastic beginner chest workout plan tailored just for you. Here are three exercises that will help build a strong and defined chest:

1. **Leverage Chest Press**:
   - This exercise is perfect for beginners because it focuses on the pectoral muscles without requiring any additional equipment beyond what you already have at home.
   - It's an excellent choice if you're looking to start building your chest strength from the ground up.

2. **Dumbbell Bench Press**:
   - The dumbbell bench press is a classic move that can be easily incorporated into most workout routines, making it perfect for beginners who want to build their chest without needing any specialized equipment.
   - It's versatile and effective, helping you develop your upper body strength in a way that feels natural.

3. **Machine Chest Fly**:
   - The machine chest fly is another excellent choice for beginners because it targets the same muscles as the dumbbell bench pre

,Title,BodyPart,Level,Type
1054,Machine chest press,Chest,Intermediate,Strength
1050,Leverage Chest Press,Chest,Beginner,Strength
1135,Dumbbell Chest Press - Gethin Variation,Chest,Intermediate,Strength
1059,Machine chest fly,Chest,Intermediate,Strength
1105,Dumbbell Bench Press,Chest,Intermediate,Strength


In [65]:
df_recipes = pd.read_csv('recipes.csv')
print("Shape:", df_recipes.shape)
print("\nColumns:", df_recipes.columns.tolist())
df_recipes.head()


Shape: (522517, 28)

Columns: ['RecipeId', 'Name', 'AuthorId', 'AuthorName', 'CookTime', 'PrepTime', 'TotalTime', 'DatePublished', 'Description', 'Images', 'RecipeCategory', 'Keywords', 'RecipeIngredientQuantities', 'RecipeIngredientParts', 'AggregatedRating', 'ReviewCount', 'Calories', 'FatContent', 'SaturatedFatContent', 'CholesterolContent', 'SodiumContent', 'CarbohydrateContent', 'FiberContent', 'SugarContent', 'ProteinContent', 'RecipeServings', 'RecipeYield', 'RecipeInstructions']


,RecipeId,Name,AuthorId,AuthorName,CookTime,PrepTime,TotalTime,DatePublished,Description,Images,...,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeYield,RecipeInstructions
0,38,Low-Fat Berry Blue Frozen Dessert,1533,Dancer,PT24H,PT45M,PT24H45M,1999-08-09T21:46:00Z,Make and share this Low-Fat Berry Blue Frozen ...,"c(""https://img.sndimg.com/food/image/upload/w_...",...,1.3,8.0,29.8,37.1,3.6,30.2,3.2,4.0,NaN,"c(""Toss 2 cups berries with sugar."", ""Let stan..."
1,39,Biryani,1567,elly9812,PT25M,PT4H,PT4H25M,1999-08-29T13:12:00Z,Make and share this Biryani recipe from Food.com.,"c(""https://img.sndimg.com/food/image/upload/w_...",...,16.6,372.8,368.4,84.4,9.0,20.4,63.4,6.0,NaN,"c(""Soak saffron in warm milk for 5 minutes and..."
2,40,Best Lemonade,1566,Stephen Little,PT5M,PT30M,PT35M,1999-09-05T19:52:00Z,This is from one of my first Good House Keepi...,"c(""https://img.sndimg.com/food/image/upload/w_...",...,0.0,0.0,1.8,81.5,0.4,77.2,0.3,4.0,NaN,"c(""Into a 1 quart Jar with tight fitting lid, ..."
3,41,Carina's Tofu-Vegetable Kebabs,1586,Cyclopz,PT20M,PT24H,PT24H20M,1999-09-03T14:54:00Z,This dish is best prepared a day in advance to...,"c(""https://img.sndimg.com/food/image/upload/w_...",...,3.8,0.0,1558.6,64.2,17.3,32.1,29.3,2.0,4 kebabs,"c(""Drain the tofu, carefully squeezing out exc..."
4,42,Cabbage Soup,1538,Duckie067,PT30M,PT20M,PT50M,1999-09-19T06:19:00Z,Make and share this Cabbage Soup recipe from F...,"""https://img.sndimg.com/food/image/upload/w_55...",...,0.1,0.0,959.3,25.1,4.8,17.7,4.3,4.0,NaN,"c(""Mix everything together and bring to a boil..."


In [66]:
drop_cols = ["AuthorId", "AuthorName", "DatePublished", "Images", 
             "RecipeIngredientQuantities", "SaturatedFatContent", 
             "CholesterolContent", "SodiumContent", "FiberContent", 
             "SugarContent"]

df_recipes = df_recipes.drop(columns=drop_cols)

RecipeId                      0
Name                          0
CookTime                  82545
PrepTime                      0
TotalTime                     0
Description                   5
RecipeCategory              751
Keywords                  17237
RecipeIngredientParts         0
AggregatedRating         253223
ReviewCount              247489
Calories                      0
FatContent                    0
CarbohydrateContent           0
ProteinContent                0
RecipeServings           182911
RecipeYield              348071
RecipeInstructions            0
dtype: int64

In [68]:
# Drop rows missing the key text columns
df_recipes = df_recipes.dropna(subset=["Description", "RecipeIngredientParts", "RecipeInstructions"])

# Sample 5000 rows
df_recipes = df_recipes.sample(n=5000, random_state=42).reset_index(drop=True)

print("Shape after cleaning:", df_recipes.shape)
print("\nMissing values:")
print(df_recipes.isnull().sum())

Shape after cleaning: (5000, 18)

Missing values:
RecipeId                    0
Name                        0
CookTime                  821
PrepTime                    0
TotalTime                   0
Description                 0
RecipeCategory              7
Keywords                  168
RecipeIngredientParts       0
AggregatedRating         2390
ReviewCount              2330
Calories                    0
FatContent                  0
CarbohydrateContent         0
ProteinContent              0
RecipeServings           1782
RecipeYield              3392
RecipeInstructions          0
dtype: int64


In [69]:
# ============================================================
# FITFUEL - Cell 12: Further Recipe Cleaning
# ============================================================

# Drop columns with too many missing values
df_recipes = df_recipes.drop(columns=["AggregatedRating", "ReviewCount", 
                                       "RecipeServings", "RecipeYield"])

# Fill remaining missing values
df_recipes["CookTime"] = df_recipes["CookTime"].fillna("Unknown")
df_recipes["RecipeCategory"] = df_recipes["RecipeCategory"].fillna("Unknown")
df_recipes["Keywords"] = df_recipes["Keywords"].fillna("Unknown")

print("Shape:", df_recipes.shape)
print("\nMissing values:")
print(df_recipes.isnull().sum())

Shape: (5000, 14)

Missing values:
RecipeId                 0
Name                     0
CookTime                 0
PrepTime                 0
TotalTime                0
Description              0
RecipeCategory           0
Keywords                 0
RecipeIngredientParts    0
Calories                 0
FatContent               0
CarbohydrateContent      0
ProteinContent           0
RecipeInstructions       0
dtype: int64


In [70]:
# ============================================================
# FITFUEL - Cell 13: Create Recipe Description Column
# ============================================================

df_recipes["description"] = (
    "Recipe: " + df_recipes["Name"] +
    " | Category: " + df_recipes["RecipeCategory"] +
    " | Ingredients: " + df_recipes["RecipeIngredientParts"] +
    " | Description: " + df_recipes["Description"] +
    " | Instructions: " + df_recipes["RecipeInstructions"]
)

# Preview one full description
print(df_recipes["description"].iloc[0])

Recipe: Hash Brown Casserole | Category: Potato | Ingredients: c("margarine", "sour cream", "cheddar cheese", "frozen hash browns") | Description: Make and share this Hash Brown Casserole recipe from Food.com. | Instructions: c("Combine soup and margarine and bring to boil.", "Add remainder of the ingredients in large mixing bowl.", "Once mixed place into a 9 x 13 casserole dish.", "Bake at 400 for 45 min to 1 hr.", "May take more time at higher altitudes.")


In [71]:
# ============================================================
# FITFUEL - Cell 14: Generate Recipe Embeddings
# ============================================================

recipe_descriptions = df_recipes["description"].tolist()

print("Generating embeddings for all recipes...")
recipe_embeddings = embedding_model.encode(recipe_descriptions, show_progress_bar=True)

print(f"\nCreated {len(recipe_embeddings)} embeddings, each with {recipe_embeddings[0].shape[0]} numbers")

Generating embeddings for all recipes...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]


Created 5000 embeddings, each with 384 numbers


In [72]:
# ============================================================
# FITFUEL - Cell 15: Build Recipe ChromaDB Collection
# ============================================================

# Delete if already exists
try:
    client.delete_collection(name="recipe_collection")
except:
    pass

# Create the collection
recipe_collection = client.get_or_create_collection(name="recipe_collection")

# Prepare data
ids        = df_recipes.index.astype(str).tolist()
documents  = df_recipes["description"].tolist()
metadatas  = df_recipes[["RecipeCategory", "Calories", "ProteinContent"]].to_dict(orient="records")
emb_list   = [emb.tolist() for emb in recipe_embeddings]

# Add in chunks (5000 rows so we need to chunk it)
for start in range(0, len(ids), 5000):
    end = start + 5000
    recipe_collection.add(
        ids        = ids[start:end],
        documents  = documents[start:end],
        metadatas  = metadatas[start:end],
        embeddings = emb_list[start:end],
    )

print(f"Loaded {recipe_collection.count()} recipes into ChromaDB.")

Loaded 5000 recipes into ChromaDB.


In [75]:
# ============================================================
# FITFUEL - Cell 16: Recipe Retrieval & Generation Functions
# ============================================================

def retrieve_recipes(query, k=5, filter_metadata=None):
    query_embedding = embedding_model.encode([query])
    results = recipe_collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k,
        where=filter_metadata if filter_metadata else None
    )
    descriptions = results["documents"][0]
    ids          = [int(i) for i in results["ids"][0]]
    rows         = df_recipes.loc[ids]
    return descriptions, rows


def generate_recipe_answer(query, context):
    prompt = (
        "You are FitFuel, a personalized nutrition assistant.\n"
        "Based ONLY on the following recipe descriptions, recommend a meal plan for the user.\n"
        "Suggest 2-3 recipes from the list and explain briefly why each fits the user's request.\n"
        "Write in an encouraging and friendly tone.\n"
        "If the recipes do not match the query at all, say 'No matching recipes found.'\n"
        "Do NOT invent recipes that are not in the list.\n\n"
        "RECIPE DESCRIPTIONS:\n" + context + "\n\n"
        "USER REQUEST: " + query
    )
    response = ollama.generate(
        model="qwen2.5:1.5b",
        prompt=prompt,
        options={"temperature": 0.1}
    )
    return response["response"]


def rag_query_recipes(query, k=5, filter_metadata=None):
    descriptions, rows = retrieve_recipes(query, k, filter_metadata)
    context = "\n\n".join(descriptions) if descriptions else "empty"
    answer = generate_recipe_answer(query, context)
    return answer, rows

# Test it!
answer, dfMatches = rag_query_recipes("I want a high protein low calorie meal", k=5)
print("FitFuel Meal Recommendation:\n", answer)
print("\nMatched recipes:")
dfMatches[["Name", "RecipeCategory", "Calories", "ProteinContent"]]

FitFuel Meal Recommendation:
 Based on your request for a high protein, low calorie meal, here are two recipes that fit the criteria:

1. **Protein Granola Bars With Kashi Golean**
   This recipe is designed to be both high in protein and low in calories. It's a compilation of various granola bar recipes with added benefits like using non-splenda products (syrup and brown sugar) for lower calorie options. The inclusion of dried fruits like cranberries, raisins, and dates adds natural sweetness without excessive sugar.

2. **Healthier Tuna Salad**
   This recipe is rich in protein thanks to the tuna and hard-boiled eggs, making it a great choice for high-protein meals. It's also low in calories due to the use of fat-free mayonnaise and reduced-fat tofu, which helps keep the nutritional value while reducing overall calorie content.

Both recipes are designed to be both nutritious and satisfying, providing you with a balanced meal that meets your dietary needs without compromising on tast

,Name,RecipeCategory,Calories,ProteinContent
2706,Protein Granola Bars With Kashi Golean,Lunch/Snacks,203.5,8.5
1581,Truly Creamy Low-Carb Reduced-Fat Mini Cheesec...,Cheesecake,286.5,14.1
3537,Healthier Tuna Salad,Lunch/Snacks,240.4,28.9
2445,Weight Watchers Healthier Egg Salad,Lunch/Snacks,65.3,3.9
3752,Babzy's Low Fat Vegetable Lasagna (Ww 7 Pts),One Dish Meal,275.4,21.6


In [76]:
# ============================================================
# FITFUEL - Cell 17: Gradio UI
# ============================================================

import gradio as gr

def fitfuel_query(query, k):
    # Run both pipelines simultaneously
    exercise_answer, exercise_rows = rag_query_exercises(query, k=int(k))
    recipe_answer, recipe_rows     = rag_query_recipes(query, k=int(k))

    # Format exercise table
    exercise_table = exercise_rows[["Title", "BodyPart", "Level", "Type"]].reset_index(drop=True)

    # Format recipe table
    recipe_table = recipe_rows[["Name", "RecipeCategory", "Calories", "ProteinContent"]].reset_index(drop=True)

    return exercise_answer, exercise_table, recipe_answer, recipe_table


with gr.Blocks(title="FitFuel - AI Fitness & Nutrition Assistant") as demo:
    gr.Markdown("""
    # 🏋️ FitFuel — Your AI Fitness & Nutrition Assistant
    Tell us your fitness goal and we'll recommend a workout AND a meal plan tailored just for you.
    """)

    with gr.Row():
        query_box = gr.Textbox(
            label="What's your fitness goal?",
            placeholder="e.g. I want to build muscle and eat high protein meals",
            lines=2
        )

    with gr.Row():
        k_slider = gr.Slider(minimum=1, maximum=10, value=5, step=1, label="Number of results to retrieve (top-k)")

    run_button = gr.Button("Get My FitFuel Plan 💪")

    gr.Markdown("## 🏋️ Workout Recommendation")
    exercise_answer_box = gr.Markdown()
    exercise_table_out  = gr.Dataframe(label="Matched Exercises", interactive=False)

    gr.Markdown("## 🥗 Meal Recommendation")
    recipe_answer_box = gr.Markdown()
    recipe_table_out  = gr.Dataframe(label="Matched Recipes", interactive=False)

    run_button.click(
        fn=fitfuel_query,
        inputs=[query_box, k_slider],
        outputs=[exercise_answer_box, exercise_table_out, recipe_answer_box, recipe_table_out]
    )

demo.launch(share=False)

Matplotlib is building the font cache; this may take a moment.


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
